# exp157_candidate_ranker_feature_enrichment train

Train-side audit for supervised PF/Beam/dense candidate selection. This extends exp101 with tvt_dense-family candidates and target-free dense disagreement features from exp072.


## 1. Setup and configuration

In [ ]:
from pathlib import Path
import json
import pandas as pd

from settings import ExperimentPaths, get_nested, load_config
from candidate_ranker_feature_enrichment import (
    DEFAULT_DENSE_FEATURE_CACHE,
    DEFAULT_DENSE_FEATURE_SCHEMA,
    DEFAULT_TRAIN_FEATURE_CACHE,
    DEFAULT_TRAIN_FEATURE_SCHEMA,
    build_required_columns,
    candidate_specs_from_config,
    find_artifact,
    run_candidate_ranker_feature_enrichment,
)

paths = ExperimentPaths()
config = load_config()
output_dir = Path('/kaggle/working/artifacts') if Path('/kaggle/working').exists() else paths.artifacts_dir
output_dir.mkdir(parents=True, exist_ok=True)

print('experiment:', get_nested(config, 'experiment.name'))
print('route:', get_nested(config, 'experiment.route'))
print('parent:', get_nested(config, 'lineage.parent'))
print('cache_parent:', get_nested(config, 'lineage.cache_parent'))
print('feature_cache_parent:', get_nested(config, 'lineage.feature_cache_parent'))
print('enable_gpu:', get_nested(config, 'runtime.kaggle.enable_gpu'))
print('output_dir:', output_dir)


## 2. Input cache audit

In [ ]:
candidates = candidate_specs_from_config(config)
required_columns = build_required_columns(config, candidates)
cache_path = find_artifact(
    DEFAULT_TRAIN_FEATURE_CACHE,
    get_nested(config, 'data.exp099_train_feature_cache_local'),
)
schema_path = find_artifact(
    DEFAULT_TRAIN_FEATURE_SCHEMA,
    get_nested(config, 'data.exp099_train_feature_schema_local'),
)
header = pd.read_csv(cache_path, nrows=0).columns.tolist()
missing = [column for column in required_columns if column not in header]
print('exp099 cache:', cache_path)
print('exp099 schema:', schema_path)
print('exp099 required columns:', len(required_columns))
print('exp099 missing required columns:', missing)
if missing:
    raise ValueError(f'exp099 cache missing required columns: {missing}')

dense_cache_path = find_artifact(
    DEFAULT_DENSE_FEATURE_CACHE,
    get_nested(config, 'data.exp072_train_feature_cache_local'),
)
dense_schema_path = find_artifact(
    DEFAULT_DENSE_FEATURE_SCHEMA,
    get_nested(config, 'data.exp072_feature_schema_local'),
)
dense_header = pd.read_csv(dense_cache_path, nrows=0).columns.tolist()
dense_required = ['id', *get_nested(config, 'ranker.feature_enrichment.auxiliary_columns')]
dense_missing = [column for column in dense_required if column not in dense_header]
print('exp072 dense cache:', dense_cache_path)
print('exp072 dense schema:', dense_schema_path)
print('exp072 required columns:', dense_required)
print('exp072 missing required columns:', dense_missing)
if dense_missing:
    raise ValueError(f'exp072 cache missing required columns: {dense_missing}')


## 3. Candidate and model plan

In [ ]:
candidate_plan = pd.DataFrame([{'candidate': spec.name, 'column': spec.column} for spec in candidates])
model_plan = pd.DataFrame([
    {'variant': 'likpf_mean_single', 'kind': 'baseline'},
    {'variant': 'multiobs_score_top1', 'kind': 'target_free_baseline'},
    {'variant': 'oracle', 'kind': 'upper_bound'},
    {'variant': 'lgb_multiclass', 'kind': 'supervised_nway_classifier'},
    {'variant': 'lgb_candidate_binary', 'kind': 'candidate_long_binary_scorer'},
    {'variant': 'lgb_candidate_error_ranker', 'kind': 'candidate_long_error_ranker'},
])
print('candidates')
display(candidate_plan)
print('model plan')
display(model_plan)
print('candidate count:', len(candidate_plan))
print('new booster count:', 3 * int(get_nested(config, 'validation.n_folds')))
print('control retraining:', False)


## 4. Run OOF ranker audit

In [ ]:
summary = run_candidate_ranker_feature_enrichment(
    output_dir=output_dir,
    cache_path=get_nested(config, 'data.exp099_train_feature_cache_local'),
    schema_path=get_nested(config, 'data.exp099_train_feature_schema_local'),
    max_rows=get_nested(config, 'ranker.max_rows'),
)
summary_path = output_dir / 'exp157_candidate_ranker_feature_enrichment_summary.json'
print('summary_path:', summary_path)
print(json.dumps(summary.get('decision', {}), indent=2, sort_keys=True))


## 5. Metrics and artifacts

In [ ]:
metrics_path = output_dir / 'exp157_candidate_ranker_feature_enrichment_metrics.csv'
dist_path = output_dir / 'exp157_candidate_ranker_feature_enrichment_selection_distribution.csv'
bucket_path = output_dir / 'exp157_candidate_ranker_feature_enrichment_bucket_metrics.csv'
importance_path = output_dir / 'exp157_candidate_ranker_feature_enrichment_feature_importance_mean.csv'

metrics = pd.read_csv(metrics_path)
display(metrics)
print('selection distribution')
display(pd.read_csv(dist_path).head(40))
print('bucket metrics')
display(pd.read_csv(bucket_path).head(40))
print('feature importance')
display(pd.read_csv(importance_path).head(60))
